## Ejercicio 7

In [8]:
import numpy as np
import matplotlib.pyplot as plt
import scipy

In [9]:
X_fantoma = np.array([
    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    [0.0, 0.0, 0.8, 0.3, 0.3, 0.3, 0.0, 0.0],
    [0.0, 0.8, 0.8, 0.8, 0.3, 0.3, 0.3, 0.0],
    [0.0, 0.3, 0.8, 0.3, 0.3, 0.3, 0.3, 0.0],
    [0.0, 0.3, 0.3, 0.3, 0.3, 0.5, 0.3, 0.0],
    [0.0, 0.3, 0.3, 0.3, 0.5, 0.5, 0.5, 0.0],
    [0.0, 0.0, 0.3, 0.3, 0.3, 0.5, 0.0, 0.0],
    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
], dtype=float)

n = X_fantoma.shape[0]
x_fantoma = X_fantoma.reshape(-1)

In [10]:
def mascara_rayo(n, direccion, s):
    alfa = direccion[0]
    beta = direccion[1]
    matriz = np.zeros((n, n), dtype=int)
    for i in range(n):
        for j in range(n):
            if alfa * i + beta * j == s:
                matriz[i][j] = 1
    return matriz


def matriz_proyeccion(n, direcciones):
    filas_A = []
    rayos = []

    for direccion in direcciones:
        alfa = direccion[0]
        beta = direccion[1]
        valores_s = set()

        for i in range(n):
            for j in range(n):
                valores_s.add(alfa * i + beta * j)

        for s in sorted(valores_s):
            mascara = mascara_rayo(n, direccion, s)
            filas_A.append(mascara.reshape(-1))
            rayos.append((direccion, s))

    A = np.array(filas_A, dtype=float)
    return A, rayos

In [11]:
D2 = [(1, 0), (0, 1)]
D4 = D2 + [(1, 1), (1, -1)]
D7 = D4 + [(2, 1), (1, 2), (3, 1)]

A2, rayos2 = matriz_proyeccion(n, D2)
A4, rayos4 = matriz_proyeccion(n, D4)
A7, rayos7 = matriz_proyeccion(n, D7)

In [12]:
b7 = A7 @ x_fantoma
rng = np.random.default_rng(67)
z = rng.standard_normal(b7.size)
escala = np.sqrt(np.mean(b7**2))

In [13]:
sigma = 0.06
b_sigma = b7 + sigma * escala * z
x_rec = np.linalg.lstsq(A7, b_sigma, rcond=None)[0]

### Inciso a)

In [14]:
r_sigma = b_sigma - A7 @ x_rec
producto = A7.T @ r_sigma
print("Norma del residuo:", np.linalg.norm(r_sigma))
print("Norma de A7.T @ r_sigma:", np.linalg.norm(producto))

Norma del residuo: 0.43834433692971386
Norma de A7.T @ r_sigma: 6.212536955172913e-14


### Inciso b)

Partimos de que la componente $j$ de $A_7^T r_\sigma$ es el producto interno entre la columna $j$ de $A_7$ y el residuo $r_\sigma$, entonces: <br>
- Como todos estos productos son aproximadamente cero, decimos que el residuo es ortogonal a todas las columnas de $A_7$ y a cualquier combinación lineal de ellas. Es decir, el residuo es ortogonal al espacio ${Col}(A_7)$.

Por otra parte, decimos que el vector $A_7\hat{x}_\sigma$ es la proyección ortogonal de $b_\sigma$ sobre ${Col}(A_7)$, ya que: <br>
- $A_7\hat{x}_\sigma$ es una combinación lineal de las columnas de $A_7$ , por lo tanto pertenece al espacio ${Col}(A_7)$. 
<br>
- Las mediciones ruidosas se pueden expresar como $b_\sigma=A_7\hat{x}_\sigma+r_\sigma \qquad r_\sigma\perp\operatorname{Col}(A_7).$

Por lo tanto, esta descomposición de las mediciones ruidosas corresponde a una proyección ortogonal: $A_7\hat{x}_\sigma$ es la proyección de $b_\sigma$ sobre el espacio columna de $A_7$, y $r_\sigma$ es la diferencia entre las mediciones originales y su proyección.

Que el residuo sea ortogonal al espacio ${Col}(A_7)$ garantiza que $A_7\hat{x}_\sigma$ es el vector de ${Col}(A_7)$ más cercano a $b_\sigma$. Por eso, la reconstrucción por mínimos cuadrados encuentra las mediciones que mejor aproximan las mediciones ruidosas.

### Inciso c)

Las **proyecciones tomográficas** son las sumas de atenuaciones obtenidas por los rayos
del escáner. Cada una produce una medición que en el modelo binario se escribe como
$b_r=a_r^Tx$.

La **proyección ortogonal de mínimos cuadrados** actúa sobre el vector completo de
mediciones en el espacio $\mathbb{R}^{119}$. Busca el punto de
$\operatorname{Col}(A_7)$ más cercano a $b_\sigma$. 